# W3C1: Loading, shaping and seeing a dataset

Run every cell from the top. **Everything already works.**

We run this together, cell by cell. Each part ends with a **TRY IT**: one question, one empty cell. The **YOUR TURN** cells at the end are the post-break work.

Today you will:

1. Refresh the **Python** everything else is built on.
2. Load 240 movie reviews with **pandas** and ask the table questions.
3. Chart them with **seaborn**.
4. Look at the **NumPy** array underneath: shape, axis, broadcasting.
5. Train a **scikit-learn** classifier on the text.

Nothing to submit. Answers are in the last cell.

In [ ]:
# Setup. Run this cell first.
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
MAROON = "#7C2529"
GRAY = "#8e8e93"

print("numpy  ", np.__version__)
print("pandas ", pd.__version__)
print("seaborn", sns.__version__)

## Part 0. The Python this is built on

Five minutes on the pieces everything else today is made of. If this is all
familiar, read it and move on; if it is not, this is the part to slow down for.

In [ ]:
# A VARIABLE is a name for one value. Nothing more.
title = "Arrival"
rating = 9

print(title, rating)
print(rating + 1)

In [ ]:
# A LIST is ORDERED, so you index it by POSITION. Counting starts at 0.
words = ["the", "film", "was", "good"]

print(words[0])       # first
print(words[-1])      # last: -1 counts back from the end
print(words[:2])      # a slice: the first two
print(len(words))     # how many

In [ ]:
# A DICTIONARY is KEYED, so you index it by NAME instead of position.
review = {"genre": "scifi", "rating": 9, "label": "pos"}

print(review["genre"])
print(list(review.keys()))

# A missing key is a CRASH, not a None. This is worth seeing once.
try:
    review["year"]
except KeyError:
    print("review['year']  ->  KeyError")

print(review.get("year"))     # .get hands back None instead of raising

In [ ]:
# A FOR LOOP repeats: start with something empty, look at each item, build up.
long_ones = []
for word in words:
    if len(word) > 3:
        long_ones.append(word)

print(long_ones)

# The counting loop, which you will meet again all term.
counts = {}
for word in ["the", "film", "was", "good", "the"]:
    counts[word] = counts.get(word, 0) + 1

print(counts)

In [ ]:
# A FUNCTION puts a name on a recipe so you can run it again on new input.
def count_long(some_words, least=4):
    """How many words are at least `least` characters long?"""
    n = 0
    for word in some_words:
        if len(word) >= least:
            n = n + 1
    return n

print(count_long(words))              # least defaults to 4
print(count_long(words, least=3))     # ...or say otherwise at the call

In [ ]:
# ================== TRY IT 0 ==================
# Use a for loop to build a dictionary mapping each word in `words`
# to its length.
# ==============================================

## Part 1. Load the data


`pd.read_csv` turns a file on disk into a **DataFrame**: a spreadsheet you can
talk to in Python. One row per review, six columns.


<img src="images/dataframe.png" width="640">

In [ ]:
# Read the file. One line.
reviews = pd.read_csv("data/reviews.csv")

# Always look at your data before anything else.
reviews.head(5)

In [ ]:
# Three questions to ask of any new table.
print("shape (rows, columns):", reviews.shape)
print()
print("column names:", list(reviews.columns))
print()
print("what type is each column?")
print(reviews.dtypes)

In [ ]:
# One column on its own is a Series. Index it by name, with square brackets.
labels = reviews["label"]

print("the first three labels:")
print(labels.head(3))
print()

# value_counts is the fastest way to see whether a dataset is balanced.
print("how many of each label?")
print(labels.value_counts())
print()

# describe() summarises the numeric columns: count, mean, min, quartiles, max.
print(reviews[["rating", "year"]].describe())

In [ ]:
# ================== TRY IT 1 ==================
# Which GENRE has the most reviews?
# ==============================================


## Part 2. Ask the table questions


Three operations cover most of what you will ever do to a DataFrame:
**select** rows, **add** a column, **group** and measure.


In [ ]:
# SELECT. A comparison on a column gives a column of True/False, one per row.
is_positive = reviews["label"] == "pos"
print("is_positive is a column of booleans, shape", is_positive.shape)
print(is_positive.head(3))
print()

# Put that boolean column inside the brackets and you keep only the True rows.
positive_reviews = reviews[is_positive]
print("kept", len(positive_reviews), "of", len(reviews), "rows")
print()
print("one of them:")
print(positive_reviews["text"].iloc[0])

In [ ]:
# ADD a column. .str gives you string operations that run down the whole column.
reviews["n_words"] = reviews["text"].str.split().str.len()
reviews["n_chars"] = reviews["text"].str.len()

print(reviews[["label", "rating", "n_words", "n_chars"]].head(5))

In [ ]:
# GROUP. Split the rows by label, then measure each group.
print("average of every numeric column, by label:")
print(reviews.groupby("label")[["rating", "n_words", "n_chars"]].mean().round(2))
print()

# Group by two columns and you get one row per combination.
print("how many reviews of each label, in each genre:")
print(reviews.groupby(["genre", "label"]).size())

In [ ]:
# ================== TRY IT 2 ==================
# What is the average star rating of each GENRE?
# Which genre comes out on top?
# ==============================================


## Part 3. See it


You cannot read 240 rows. You can read a chart of them in one second.

Name the columns for `x`, `y` and `hue`. Seaborn does the grouping and the
counting itself.


<img src="images/seaborn-idea.png" width="640">

In [ ]:
# COUNTPLOT: how many rows fall in each category. No y, seaborn counts for you.
plt.figure(figsize=(6, 3.5))
sns.countplot(data=reviews, x="genre", hue="label", palette=[GRAY, MAROON])
plt.title("Reviews per genre, split by label")
plt.show()

In [ ]:
# HISTPLOT: the shape of one numeric column. Here, how long reviews are.
plt.figure(figsize=(6, 3.5))
sns.histplot(data=reviews, x="n_words", hue="label", bins=14,
             palette=[GRAY, MAROON], multiple="stack")
plt.title("How long is a review?")
plt.xlabel("words in the review")
plt.show()

In [ ]:
# BOXPLOT: a whole distribution per group. The line is the median, the box holds
# the middle half of the values, the whiskers reach the rest.
plt.figure(figsize=(5, 3.5))
sns.boxplot(data=reviews, x="label", y="rating", hue="label",
            palette=[GRAY, MAROON], legend=False)
plt.title("Star rating by label")
plt.show()

print("The labels and the ratings agree, which is a sanity check on the data.")
print(reviews.groupby("label")["rating"].median())

In [ ]:
# BARPLOT: one number per group, with a confidence interval drawn for free.
plt.figure(figsize=(6, 3.5))
sns.barplot(data=reviews, x="genre", y="n_words", hue="label",
            palette=[GRAY, MAROON])
plt.title("Average review length by genre and label")
plt.ylabel("words per review")
plt.show()

In [ ]:
# ================== TRY IT 3 ==================
# Draw a histogram of the star rating, coloured by label.
# Which separates the two labels better, the rating or the review length?
# ==============================================


## Part 4. Underneath the table is a NumPy array


A DataFrame is a wrapper. The numbers live in a NumPy array, and every model in
this course works on arrays.

The idea to carry out of today is **shape**: `(240, 2)` is 240 rows of 2 numbers.


<img src="images/shape-axis.png" width="640">

In [ ]:
# .to_numpy() drops the column names and hands back the raw block of numbers.
numbers = reviews[["rating", "n_words"]].to_numpy()

print("type :", type(numbers).__name__)
print("shape:", numbers.shape, " -> 240 reviews, 2 numbers each")
print("dtype:", numbers.dtype)
print()
print("the first three rows:")
print(numbers[:3])

In [ ]:
# AXIS. Every reduction takes an axis, and the axis you name is the one that
# disappears. Print the shapes and it stops being mysterious.
print("numbers.shape        ", numbers.shape)
print("mean(axis=0).shape   ", numbers.mean(axis=0).shape, " <- collapses the 240 rows")
print("mean(axis=1).shape   ", numbers.mean(axis=1).shape, " <- collapses the 2 columns")
print()
print("mean of each column:", numbers.mean(axis=0).round(2), " (average rating, average length)")
print("mean of first row  :", numbers.mean(axis=1)[0].round(2), " (meaningless: it averages a rating with a word count)")

In [ ]:
# BROADCASTING. NumPy stretches a small array to fit a big one, so you never
# write the loop. Here we put both columns on the same scale.
column_means = numbers.mean(axis=0)     # shape (2,)
column_stds = numbers.std(axis=0)       # shape (2,)

scaled = (numbers - column_means) / column_stds   # (240, 2) - (2,) -> (240, 2)

print("numbers", numbers.shape, "-", column_means.shape, "->", scaled.shape)
print()
print("after scaling, each column has mean 0 and standard deviation 1:")
print("  means:", scaled.mean(axis=0).round(6))
print("  stds :", scaled.std(axis=0).round(6))

In [ ]:
# MASKING. The same True/False trick as pandas, on the raw array.
ratings = numbers[:, 0]          # every row, column 0
lengths = numbers[:, 1]          # every row, column 1

is_long = lengths > 20           # a boolean array, shape (240,)

print("ratings.shape:", ratings.shape)
print(is_long.sum(), "reviews are longer than 20 words")
print("their average rating:", ratings[is_long].mean().round(2))
print("everyone else's     :", ratings[~is_long].mean().round(2), " (~ means NOT)")
print()
print("longest review is number", lengths.argmax(), "with", int(lengths.max()), "words")

In [ ]:
# ================== TRY IT 4 ==================
# Using `numbers`, how many stars and how many words are there in the
# whole corpus?
# ==============================================


## Part 5. Text in, label out


`CountVectorizer` turns the reviews into a matrix: one row per review, one column
per word, each entry a count. `MultinomialNB` is Naive Bayes, already written.

Split first. The model learns from the training rows and is judged on rows it has
never seen.


<img src="images/sklearn-idea.png" width="640">

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# SPLIT. stratify keeps the pos/neg balance the same in both halves.
text_train, text_test, y_train, y_test = train_test_split(
    reviews["text"], reviews["label"],
    test_size=0.25, random_state=0, stratify=reviews["label"])

print("training on", len(text_train), "reviews, testing on", len(text_test))

In [ ]:
# VECTORIZE. fit_transform LEARNS the vocabulary and converts, in one call.
# The test set only gets transform: it must use the vocabulary already learned.
vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(text_train)
X_test = vectorizer.transform(text_test)

vocabulary = vectorizer.get_feature_names_out()

print("X_train.shape:", X_train.shape, " -> 180 reviews x", X_train.shape[1], "distinct words")
print("X_test.shape :", X_test.shape, "  <- same number of columns, by construction")
print()
print("the first ten words of the vocabulary:", list(vocabulary[:10]))

In [ ]:
# TRAIN and SCORE. Two lines each.
model = MultinomialNB()
model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"accuracy on the 60 unseen reviews: {accuracy:.3f}")
print()
print(classification_report(y_test, predictions))

In [ ]:
# WHERE IT WENT WRONG. A confusion matrix counts every (true, predicted) pair,
# and a heatmap makes it readable at a glance.
matrix = confusion_matrix(y_test, predictions, labels=["neg", "pos"])

plt.figure(figsize=(4.2, 3.6))
sns.heatmap(matrix, annot=True, fmt="d", cmap="Reds", cbar=False,
            xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
plt.xlabel("predicted")
plt.ylabel("true")
plt.title(f"Naive Bayes, accuracy {accuracy:.3f}")
plt.show()

print("counts:", matrix.tolist())

In [ ]:
# READ THE MISTAKES. This is the habit worth building: look at what it got wrong.
wrong = pd.DataFrame({"text": text_test, "true": y_test, "predicted": predictions})
wrong = wrong[wrong["true"] != wrong["predicted"]]

print(len(wrong), "reviews were misclassified. Here are three:")
print()
for row_number in range(3):
    row = wrong.iloc[row_number]
    print(f"  true {row['true']}, predicted {row['predicted']}")
    print(f"    {row['text']}")
    print()

In [ ]:
# WHAT THE MODEL LEARNED. The difference between the two classes' log
# probabilities scores every word from most negative to most positive.
scores = model.feature_log_prob_[1] - model.feature_log_prob_[0]
word_scores = pd.DataFrame({"word": vocabulary, "score": scores})

most_positive = word_scores.sort_values("score", ascending=False).head(10)
most_negative = word_scores.sort_values("score").head(10)
extremes = pd.concat([most_negative, most_positive])

plt.figure(figsize=(6, 5))
colors = []
for value in extremes["score"]:
    if value > 0:
        colors.append(MAROON)
    else:
        colors.append(GRAY)
sns.barplot(data=extremes, y="word", x="score", palette=colors, hue="word", legend=False)
plt.title("Most negative and most positive words")
plt.xlabel("log P(word | pos) - log P(word | neg)")
plt.show()

In [ ]:
# ================== TRY IT 5 ==================
# 0.850 was measured on the 60 reviews the model had never seen.
# What does it score on the 180 it learned from, and why is that not the
# number you report?
# ==============================================



---

## Your turn

Two tasks, each a few lines and a chart. Alone or with the person next to you.


In [ ]:
# ================== YOUR TURN 1 ==================
# Naive Bayes got 0.850. Does LogisticRegression do better on the same
# features? Change the marked line: same .fit and .predict, so nothing else
# moves.
#
# Hint: LogisticRegression(max_iter=1000)
#
# Expected: accuracy goes UP, 0.850 -> 0.917, and the heatmap's off-diagonal
#           cells drop from 5 and 4 to 4 and 1.
# =================================================
from sklearn.linear_model import LogisticRegression

second_model = MultinomialNB()          # <-- change this line
second_model.fit(X_train, y_train)

second_predictions = second_model.predict(X_test)
second_accuracy = accuracy_score(y_test, second_predictions)
print(f"accuracy: {second_accuracy:.3f}   (Naive Bayes got {accuracy:.3f})")

second_matrix = confusion_matrix(y_test, second_predictions, labels=["neg", "pos"])
plt.figure(figsize=(4.2, 3.6))
sns.heatmap(second_matrix, annot=True, fmt="d", cmap="Reds", cbar=False,
            xticklabels=["neg", "pos"], yticklabels=["neg", "pos"])
plt.xlabel("predicted")
plt.ylabel("true")
plt.title(f"second model, accuracy {second_accuracy:.3f}")
plt.show()

In [ ]:
# ================== YOUR TURN 2 ==================
# Now change the FEATURES instead of the classifier. Fill in the two
# None entries and run the cell:
#
#     CountVectorizer(stop_words="english")     drop "the", "and", "is", ...
#     CountVectorizer(ngram_range=(1, 2))       count word PAIRS too
#
# Expected: three bars. Dropping stopwords helps a little, 0.850 -> 0.867.
#           Bigrams HURT, 0.850 -> 0.833: they triple the vocabulary without adding
#           evidence, so every count gets thinner.
# =================================================
setups = {
    "words": CountVectorizer(),
    "no stopwords": None,          # <-- fill this in
    "words + pairs": None,         # <-- and this one
}

results = []
for name, this_vectorizer in setups.items():
    if this_vectorizer is None:
        print(f"{name}: not yet")
        continue
    A_train = this_vectorizer.fit_transform(text_train)
    A_test = this_vectorizer.transform(text_test)
    this_model = MultinomialNB()
    this_model.fit(A_train, y_train)
    this_accuracy = accuracy_score(y_test, this_model.predict(A_test))
    results.append({"features": name, "accuracy": this_accuracy,
                    "vocabulary size": A_train.shape[1]})
    print(f"{name}: {this_accuracy:.3f}  ({A_train.shape[1]} columns)")

scoreboard = pd.DataFrame(results)
if len(scoreboard) == 3:
    plt.figure(figsize=(6, 3.5))
    sns.barplot(data=scoreboard, x="features", y="accuracy",
                hue="features", palette=[GRAY, MAROON, GRAY], legend=False)
    plt.ylim(0.75, 0.95)
    plt.title("Same classifier, three feature sets")
    plt.show()
    print(scoreboard)

## Answers

Try each task before reading.

In [ ]:
# TRY IT 0
#   lengths = {}
#   for word in words:
#       lengths[word] = len(word)
#   lengths
#   {'the': 3, 'film': 4, 'was': 3, 'good': 4}

# TRY IT 1
#   reviews["genre"].value_counts()
#   horror 67, comedy 59, scifi 59, drama 55.

# TRY IT 2
#   reviews.groupby("genre")["rating"].mean()
#   comedy 4.64, drama 5.60, horror 5.85, scifi 5.37. With 55 to 67 reviews per
#   genre, a one-star gap is not much more than noise.

# TRY IT 3
#   sns.histplot(data=reviews, x="rating", hue="label", palette=[GRAY, MAROON])
#   plt.show()
#   Two humps that never overlap: neg is 1 to 5, pos is 6 to 10. Length does not
#   separate the labels at all; rating separates them completely.

# TRY IT 4
#   numbers.sum(axis=0)
#   axis=0 collapses the 240 rows, leaving one number per column: [1291 4333].

# TRY IT 5
#   accuracy_score(y_train, model.predict(X_train))
#   0.894 on the training reviews against 0.850 on the test reviews.

# YOUR TURN 1
#   second_model = LogisticRegression(max_iter=1000)
#
#   0.850 -> 0.917. Naive Bayes assumes every word is independent evidence, so a
#   review that praises the film for two sentences and pans it in the last one
#   drowns the verdict. Logistic regression weighs the words against each other
#   instead of counting them separately, and it recovers three of those.

# YOUR TURN 2
#   "no stopwords":  CountVectorizer(stop_words="english")
#   "words + pairs": CountVectorizer(ngram_range=(1, 2))
#
#   words          0.850   221 columns
#   no stopwords   0.867   157 columns
#   words + pairs  0.833   674 columns
#
#   More features is not better. Bigrams could in principle catch "not worth",
#   but with 180 training reviews there is not enough data to estimate 674
#   columns, so the extra columns are noise.

# The three things worth carrying out of today:
#   1. Look at the data before you model it. head(), shape, value_counts(), a chart.
#   2. shape tells you what an array means, and the axis you name is the one
#      that disappears.
#   3. Read the mistakes, not just the accuracy.